# CELLULAR AUTOMATA (2d)

## What are Cellular Automata?

Cellular automata (CA) are simple computational systems made of:

* a grid of cells
* a finite set of states (e.g., alive/dead)
* local update rules

Each cell evolves over discrete time steps according to a rule that depends only on:

* its current state
* the state of its neighbors

Despite their simplicity, cellular automata can generate surprisingly complex and emergent behavior.

---

## From 1D to 2D

In **1D cellular automata**, cells are arranged in a line.
In **2D cellular automata**, cells live on a grid (like pixels in an image).

This makes them:

* more visual
* more expressive
* widely used in simulations and games

A 2D CA can be seen as a matrix that evolves over time.

---

## Basic Components of a 2D CA

### 1. Grid

A 2D lattice of cells, usually represented as a matrix:

Each cell holds a state (often binary: 0 or 1).

---

### 2. Neighborhood

A key concept is how we define neighbors.

Two common neighborhoods:

**Moore neighborhood (8 neighbors)**
Includes diagonals.

**Von Neumann neighborhood (4 neighbors)**
Only up, down, left, right.

```text
Moore (8)         Von Neumann (4)
⬛⬛⬛               ⬜⬛⬜
⬛🟥⬛               ⬛🟥⬛
⬛⬛⬛               ⬜⬛⬜
```

The choice of neighborhood strongly affects the dynamics.

---

### 3. States

Cells usually have:

* binary states (alive/dead)
* small discrete states (e.g., 0–3)
* sometimes continuous values

Binary automata are the most common and easiest to explore.

---

### 4. Update Rule

The rule determines how the grid evolves.

It maps:

```
(current cell, neighbors) → next state
```

Rules are applied:

* synchronously
* at discrete time steps

This creates a sequence of grid states over time.

---

## Example: Conway's Game of Life

One of the most famous 2D cellular automata.

Rules:

* A live cell survives with 2 or 3 live neighbors
* A dead cell becomes alive with exactly 3 neighbors
* Otherwise, the cell dies or stays dead

Even with such simple rules, the system produces:

* oscillators
* moving patterns (gliders)
* chaotic structures

---

## Why Are Cellular Automata Interesting?

Cellular automata are a classic example of:

> Complex behavior emerging from simple local rules.

They are used in:

* artificial life
* physics simulations
* procedural generation
* epidemiological models
* games and interactive systems

They also provide intuition about:

* emergence
* self-organization
* decentralized systems

---

## Visualizing Evolution

A cellular automaton is best understood dynamically.

Typical workflow in notebooks:

1. Initialize a random grid
2. Apply the rule iteratively
3. Visualize each step

---

## Key Properties

Some important characteristics of 2D cellular automata:

* **Locality** — updates depend only on nearby cells
* **Parallelism** — all cells update simultaneously
* **Determinism** — same initial state → same evolution
* **Emergence** — complex patterns arise naturally

---

In this work rules follow the format **ALIVE / BORN / STATE / NEIGHBOURHOOD**

In [ ]:
import pygame 
import numpy as np

pygame 2.5.2 (SDL 2.28.3, Python 3.12.3)
Hello from the pygame community. https://www.pygame.org/contribute.html


## Moore Neighborhood

|M |M |M| 
|-|-|-|
|**M** |:) |**M** |
|**M** |**M** |**M** |

In [3]:
def moore_neighbourhood(grid, index):
    y = index[0]
    x = index[1]

    side = len(grid)
    moore = []
    for i in range(y-1, y+2):
        k = i%side
        row = []
        for j in range (x-1, x + 2):
            w = j%side
            row.append(grid[k, w])
        moore.append(row)
    
    moore = np.array(moore)
    offset = 1 if grid[index] !=0 else 0 # non dobbiamo considerare la cellula stessa
    return np.count_nonzero(moore) - offset 

## Von Neumann Neighborhood

| |M | | 
|-|-|-|
|**M** |:) |**M** |
| |**M** | |

In [4]:
def von_neighbourhood(grid, index):
    y = index[0]
    x = index[1]

    side = len(grid)
    von = []
    
    for i in range (y-1, y+2):
        k = i%side
        if y != k:
            von.append(grid[k, x])
    
    for j in range (x-1, x+2):
        w = j%side
        if x != w:
            von.append(grid[y, w])
    
    von = np.array(von)
    return np.count_nonzero(von)

## Diagonal Neighbourhood

|M |  |M| 
|-|-|-|
| |:) | |
|**M** | |**M** |

In [5]:
def diag_neighbourhood(grid, index):
    return moore_neighbourhood(grid, index) - von_neighbourhood(grid, index)

In [6]:
g = np.array([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]])
print(g)
von_neighbourhood(g, (2, 1))


[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]
 [13 14 15 16]]


4

## Rule class

In [7]:
class Rule:
    
    def __init__(self, alive, born, state, neighbourhood):
        self.alive = alive
        self.born = born
        self.state = state
        self.neighbourhood = neighbourhood

    def update_cell(self, grid, index):
        pass

    def update_grid(self, grid):
        pass

In [8]:
class CArule(Rule):

    def __init__(self, alive, born, state, neighbourhood):
        super().__init__(alive, born, state, neighbourhood)

    def update_cell(self, grid, index):
        if grid[index] == 1:
            if self.neighbourhood(grid, index) in self.alive:
                return 1
            else:
                return -self.state
        elif grid[index] == 0:
            if self.neighbourhood(grid, index) in self.born:
                return 1
            else:
                return 0
        else:
            return grid[index] + 1
        
    def update_grid(self, grid):
        side = len(grid)
        new_grid = np.zeros_like(grid)
        for i in range (side):
            for j in range(side):
                new_grid[i, j] = self.update_cell(grid, (i, j))
        return new_grid

## Principal components

1. The **grid**
2. The **rule**

In [9]:
side = 100
grid = np.zeros(shape=(side, side))

In [15]:
rule = CArule([1, 2, 3], [1, 2, 3], 5, moore_neighbourhood)

This is a simple graphic interface to select the initial alive cells.

In [22]:
pygame.init()

side = 100
grid = np.zeros(shape=(side, side))

window_size = 600
cell_dimension = window_size//side
background_color = (0, 0, 0)
line_color = (255, 255, 255)

fps = 60

screen = pygame.display.set_mode((window_size, window_size))
pygame.display.set_caption("Cellular Automata - Setup")

clock = pygame.time.Clock()

running = True

while running:
    
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            pygame.quit()
            running = False
            
        elif event.type == pygame.MOUSEBUTTONDOWN:
            x, y = event.pos[0]//cell_dimension, event.pos[1]//cell_dimension
            if 0 <= x < side and 0 <= y < side:
                grid[y, x] = 0 if grid[y, x] == 1 else 1
    
    if running:

        screen.fill(background_color)

        for row in range(side):
            for column in range(side):
                x = column * cell_dimension
                y = row * cell_dimension
                cell_color = (0, 0, 0) if grid[row, column] == 0 else (255, 255, 255)
                pygame.draw.rect(screen, cell_color, (x, y, cell_dimension, cell_dimension)) # questo è per la cella attuale
                pygame.draw.rect(screen, line_color, (x, y, cell_dimension, cell_dimension), 1) # questo è per i bordi

        pygame.display.flip()

        clock.tick(fps)

And this is the graphic interface.

In [21]:
pygame.init()

window_size = 600
cell_dimension = window_size//side
line_color = (255, 255, 255)

light_theme = False
background_color = (255, 255, 255) if light_theme else (0, 0, 0)

fps = 5

screen = pygame.display.set_mode((window_size, window_size))
pygame.display.set_caption("Cellular Automata")

clock = pygame.time.Clock()

running = True

while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            pygame.quit()
            running = False

    if running:

        for row in range(side):
            for column in range(side):
                x = column * cell_dimension
                y = row * cell_dimension
                k = grid[row, column]
                
                # light theme
                if light_theme:
                    shade = 255 if k == 1 else 255*(((rule.state + k + 1)/rule.state))
                    cell_color = (255, 255, 255) if k == 0 else (shade, shade, 255)
                # dark theme
                else:
                    shade = 255 if k == 1 else 255*(1 - ((rule.state + k + 1)/rule.state))
                    cell_color = (0, 0, 0) if k == 0 else (0, 0, shade)

                pygame.draw.rect(screen, cell_color, (x, y, cell_dimension, cell_dimension)) # questo è per la cella attuale
                #pygame.draw.rect(screen, line_color, (x, y, cell_dimension, cell_dimension), 1) # questo è per i bordi
        
        grid = rule.update_grid(grid)

        pygame.display.flip()

        clock.tick(fps)
